# TTS bootcamp on Kaggle: Task 4 (speed-up) timing + audio

Before running: **Settings → Accelerator = GPU T4**, **Internet = On** (needs phone-verified account),
and **Add-ons → Secrets → HF_TOKEN** (your Hugging Face token, with the IndicF5 terms accepted).
Attach the dataset made from `tts_kit.zip` (Add Input). Then **Run All**. Every heavy step runs in its own
subprocess, so no kernel restart is needed.

In [ ]:
import os
os.environ["HF_TOKEN"] = "__HF_TOKEN__"

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

src = next(p.parent for p in Path("/kaggle/input").rglob("metrics.py"))
KIT = Path("/kaggle/working/tts_kit")
if not KIT.exists():
    shutil.copytree(src, KIT)
os.chdir(KIT)
print("kit:", KIT, "->", sorted(os.listdir(KIT)))

try:
    if os.environ.get("HF_TOKEN", "").startswith("hf_"):
        print("HF_TOKEN already set in this session")
    else:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("Could not read the HF_TOKEN secret (" + type(e).__name__ + "). Add it under Add-ons > Secrets and tick it,")
    print("or paste the token below for this session only (it is not saved anywhere).")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN missing or malformed"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
Path("out").mkdir(exist_ok=True)
t3 = next(p.parent for p in Path("/kaggle/input").rglob("en2dev.py"))
(KIT / "code").mkdir(exist_ok=True)
shutil.copy(t3 / "en2dev.py", KIT / "code" / "en2dev.py")
shutil.copy(t3 / "t3_idea_text.csv", KIT / "predictions" / "t3_idea_text.csv")
for f in ["heldout_L.csv", "heldout_D.csv"]:
    shutil.copy(t3 / f, KIT / "data" / f)
print("Task 3 files copied from", t3)

def sh(cmd, retry_on=None, show=12000):
    """Run a command in a fresh process; if it asks for the one-time length-fix restart, simply run it again."""
    print("$", cmd[:200])
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode and retry_on and retry_on in out:
        print("(length fix was just applied: running again, this happens once)")
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        out = (r.stdout or "") + (r.stderr or "")
    print(out[-show:])
    if r.returncode:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd[:120]}")
    return out

In [ ]:
print(sh("nvidia-smi -L"))
print(sh("python -c \"import torch,sys;print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())\""))

## 1. Install (numpy pinned as the kit says; IndicF5 from GitHub main with --no-deps because its setup.py wants numpy<=1.26)

In [ ]:
sh("pip install -q 'numpy>=2.1,<2.3' 'scipy>=1.15,<2' 'transformers==4.49.0' accelerate soundfile librosa pyyaml pandas 'speechbrain>=1.0' huggingface_hub")
sh("pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7")
sh("pip install -q cached_path click datasets ema_pytorch hydra-core jieba matplotlib pydub pypinyin safetensors tomli torchdiffeq tqdm transformers_stream_generator vocos x_transformers")
sh("python -c \"import numpy,transformers;print('numpy',numpy.__version__,'transformers',transformers.__version__)\"")

## 2. Baseline timing (baseline text, pinned settings, 24 validation items), same session as the fast runs

In [ ]:
sh("python baseline/run_baseline.py --items data/val_items.csv --out predictions/val_base_t4 --seed 0", retry_on="length fix")
base = json.load(open("predictions/val_base_t4_timing.json"))
print("baseline synthesis seconds:", base["total_synth_s"], base["gpu"])

## 3. Task 3 system with fewer flow-matching steps (nfe_step), validation: time + CER for each

In [ ]:
REF_CER = 0.0582            # val_idea CER of the Task 3 system (nfe_step 32)
LIMIT = 1.2 * REF_CER
rows = []
for n in [16, 12, 8]:
    sh(f"python baseline/run_baseline.py --items data/val_items.csv --out predictions/val_fast_n{n} "
       f"--text-csv predictions/t3_idea_text.csv --text-column idea_text --seed 0 --set nfe_step={n}", retry_on="length fix")
    out = sh(f"python metrics.py predictions/val_fast_n{n} --json out/val_fast_n{n}.json --csv out/val_fast_n{n}_scores.csv")
    Path(f"out/metrics_val_fast_n{n}.log").write_text(out, encoding="utf-8")
    t = json.load(open(f"predictions/val_fast_n{n}_timing.json"))["total_synth_s"]
    m = json.load(open(f"out/val_fast_n{n}.json"))
    rows.append((n, t, m["cer"], m["sim"], m["failed"]))
    print("nfe", n, "time", t, "CER", m["cer"], "SIM", m["sim"], "failed", m["failed"])
print("baseline", base["total_synth_s"], "limit CER", round(LIMIT, 4))
ok = [r for r in rows if r[2] <= LIMIT and r[4] == 0]
best = min(ok, key=lambda r: r[1]) if ok else None
print("table (nfe, seconds, cer, sim, failed):", rows)
print("chosen by code:", best)
json.dump({"rows": rows, "baseline_s": base["total_synth_s"], "limit": LIMIT, "chosen": best}, open("out/t4_sweep.json", "w"))

## 4. Evaluation audio with the chosen setting -> `t4_fast/`

In [ ]:
assert best, "no setting passed the guard; Task 4 not submitted"
n = best[0]
sh("python baseline/run_baseline.py --items data/eval_items.csv --out predictions/t4_fast "
   f"--text-csv predictions/t3_idea_text.csv --text-column idea_text --seed 0 --set nfe_step={n}", retry_on="length fix")
print(len(list(Path("predictions/t4_fast").glob("*.wav"))), "wav files in predictions/t4_fast")

## 5. Bundle everything to download

In [ ]:
import zipfile
zp = Path("/kaggle/working/tts_outputs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["predictions/t4_fast/*.wav", "predictions/val_fast_n*/*.wav", "predictions/*_timing.json", "out/*"]:
        for p in Path(".").glob(pat):
            if p.is_file():
                z.write(p, p.as_posix())
print(zp, round(zp.stat().st_size / 1e6, 1), "MB")